# HANDS-ON 01 | Primeros pasos con Apache Spark y PySpark
**Curso:** Base de Datos Avanzadas y Big Data · **Unidad III:** Big Data · **Semana:** 11  
**Docente:** Dr. Eric Gustavo Coronel Castillo · **Duración:** 4 horas  
**Estudiante:** _________________________________________________

> **Propósito:** poner en funcionamiento Spark, leer datos y diagnosticar su calidad antes de proponer un análisis. Las celdas señaladas **RETO** no contienen la solución.

**Datos:** `registros_operativos_simulados.csv` (28 filas ficticias creadas exclusivamente para aprendizaje; no son mediciones oficiales).

## 0. Preparación del entorno
Se sugiere **Google Colab**. Ejecuta la celda siguiente una sola vez por sesión y reinicia el entorno de ejecución **si el instalador lo indica**. Si trabajas en Jupyter local con PySpark instalado, omite la instalación.

Referencia: https://spark.apache.org/docs/latest/api/python/getting_started/install.html

In [ ]:
# Ejecuta esta celda solo si estas en Google Colab
# Solo una vez al inicio de la sesión

%pip -q install pyspark

### Comprobación de software
Si el inicio falla, registra el mensaje completo, la versión de Python y de Java. En ejecución local, Apache Spark requiere una instalación Java compatible con la versión elegida.

In [ ]:
import sys
import pyspark

print("Python:",sys.version.split()[0])
print("PySpark:",pyspark.__version__)

## 1. Reto de inicio: ¿qué preguntas debemos responder?
Una organización ficticia registra cumplimiento de metas diarias en plantas de agua, energía y residuos. Se sospecha que existen errores de captura de datos. Antes de generar un indicador gerencial, propón **dos preguntas verificables** y **dos reglas de calidad**.

**Respuesta del equipo:**
- Pregunta 1: ...
- Pregunta 2: ...
- Regla 1: ...
- Regla 2: ...

## 2. Crear una sesión Spark (ejemplo guiado)
`local[*]` utiliza los recursos disponibles de la máquina en un modo local; **no significa que se haya desplegado un clúster de varias máquinas**.

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("local[*]")
         .appName("UPN_BigData_S11_Intro")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Modo:", spark.sparkContext.master)

## 3. Cargar el archivo CSV
**Colab:** usa el panel lateral Archivos para subir el CSV; normalmente quedará en `/content/`. **Jupyter local:** coloca el CSV en la misma carpeta del notebook. El siguiente bloque detecta ambas opciones.

In [ ]:
from pathlib import Path

# Rutas candidatas
candidatos = [
    Path("registros_operativos_simulados.csv"),
    Path("/content/registros_operativos_simulados.csv")
]

# Encontrar la primera ruta existente
ruta = next((str(x) for x in candidatos if x.exists()), None)
assert ruta, "No se encontró el CSV. Súbelo en Colab o colócalo junto al notebook."
print("Archivo localizado:", ruta)

# Lectura del archivo CSV y cargar su contenido en un DataFrame de Spark.
df = (spark.read
    .option("header", True)       # Indica que la primera fila contiene los nombres de las columnas.
    .option("inferSchema", True)  # Spark determina automaticamente los tipos de datos.
    .option("encoding", "UTF-8")  # Especifica la codificacion de los caracteres del archivo.
    .csv(ruta)                    # Lee el archivo CSV ubicado en la ruta indicada.
)

# Exploracion basica de los datos
df.printSchema()  # Estructura del DataFrame.
df.show(8, truncate=False)  # Muestra las primeras 8 filas.
print("Filas cargadas:", df.count())  # Cuenta todas las filas.


**Analiza:** ¿Qué tipos de dato asignó Spark a `meta_diaria` y `valor_observado`? ¿Qué riesgos implica confiar siempre en la inferencia de esquemas?

## 4. Demostración: seleccionar y filtrar (ejemplo guiado)
Prueba con transformaciones y una acción, e identifica qué se ejecuta en cada línea.
- **filter()**: Transformacion de filtrado, permite seleccionar filas.
- **select()**: Transformacion de seleccion, permite seleccionar las columnas que necesitamos.
- **show()**: Es una accion (action), en este caso solicita ejecutar las transformaciones necesarias y muestra los resultados.

Plan de ejecucion:
```
          DATAFRAME ORIGINAL
                  |
                  v
       filter(zona == "Norte")
                  |
                  v
       select(id_registro,
              sector,
              meta_diaria)
                  |
                  v
         PLAN DE EJECUCIÓN
                  |
                  v
          show(5, False)
                  |
                  v
       EJECUCIÓN EN SPARK
                  |
                  v
         MOSTRAR RESULTADO
```


In [ ]:
df_norte = df.filter(df.zona == "Norte").select("id_registro","sector","meta_diaria")
print("Se definió una transformación")
df_norte.show(5, truncate=False)

### Spark utiliza evaluacion diferida
Analiza el siguiente codigo.

In [ ]:
# Primera transformación
df_filtrado = df.filter(df.zona == "Norte")

# Segunda transformación
df_norte = df_filtrado.select(
    "id_registro",
    "sector",
    "meta_diaria"
)

print("Las transformaciones han sido definidas.")

# Ejecutar una acción
df_norte.show(5, truncate=False)

## 5. RETO A: diagnóstico de calidad
Completa las celdas para identificar:
- (1) valores nulos.
- (2) metas negativas.
- (3) identificadores repetidos.

No elimines registros automáticamente: explica primero el criterio de negocio.

**Evidencia:** tabla de hallazgos con cantidad, IDs afectados y recomendación.

### Funciones importadas de PySpark

La siguiente instrucción importa funciones que permiten trabajar con
columnas y realizar operaciones de agregación en DataFrames de Spark.

~~~python
from pyspark.sql.functions import col, count, sum as spark_sum, avg
~~~

| Función | Descripción | Ejemplo de uso |
|---|---|---|
| `col()` | Permite referenciar una columna del DataFrame por su nombre. | `col("valor_observado")` |
| `count()` | Cuenta valores no nulos de una columna. Con `count("*")` cuenta todas las filas. | `count("valor_observado")` |
| `spark_sum()` | Calcula la suma de los valores no nulos de una columna. | `spark_sum("meta_diaria")` |
| `avg()` | Calcula el promedio de los valores no nulos de una columna. | `avg("valor_observado")` |

**Observaciones:**

- `col()` permite construir expresiones para filtrar, seleccionar o transformar columnas.
- `count()` permite realizar agregaciones para determinar cantidades de registros o valores.
- `spark_sum()` es un alias de la función `sum()` de PySpark. Se utiliza para evitar conflictos con la función `sum()` incorporada en Python.
- `avg()` calcula el promedio aritmético, excluyendo los valores `NULL`.
- Estas funciones pertenecen al módulo `pyspark.sql.functions`.

In [ ]:
# RETO A.1: cuenta registros con valor observado nulo
# Pista: from pyspark.sql.functions import col
# Escribe tu código debajo:

# SOLUCIÓN A.1 — identificar filas sin valor observado.
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round
faltantes = df.filter(col("valor_observado").isNull())
faltantes.select("id_registro", "sector", "valor_observado").show(truncate=False)
# Nota: .isNull() identifica NULL; no equivale a cadenas vacías en columnas de texto.


In [ ]:
# RETO A.2: encuentra filas con meta_diaria <= 0
# Escribe tu código debajo:

# SOLUCIÓN A.2 — separar metas inválidas de observaciones atípicas.
metas_invalidas = df.filter(col("meta_diaria") <= 0)
metas_invalidas.select("id_registro", "meta_diaria", "valor_observado").show(truncate=False)
# R026 requiere verificación; no se debe cambiar -20 a +20 sin trazabilidad.


In [ ]:
# RETO A.3: identifica IDs duplicados
# Pista: groupBy(...).count().filter(...)
# Escribe tu código debajo:

# SOLUCIÓN A.3 — detectar identificadores repetidos.
duplicados = (
    df.groupBy("id_registro")
    .count()
    .filter(col("count") > 1)
)
duplicados.show(truncate=False)


In [ ]:
# Cambiando el nombre de la columna

duplicados2 = (
    df.groupBy("id_registro")
    .count()
    .withColumnRenamed("count", "cantidad")
    .filter(col("cantidad") > 1)
)
duplicados2.show(truncate=False)


**Pregunta de decisión:** `R027` tiene un valor observado mayor que su meta. ¿Es necesariamente un error? Sustenta una regla de validación que distinga inconsistencia técnica de fenómeno legítimo.

## 6. RETO B: exploración y comparación
Propón una consulta en la API de DataFrames que permita comparar registros por sector. Primero define qué métrica utilizarás y qué registros excluirás (si corresponde). Luego presenta un resultado reproducible.

**Evita** interpretar las estadísticas como hechos reales: todos los datos son simulados.

In [ ]:
# RETO B: agrupación por sector
# Pista: df.groupBy("sector").count()
# Amplía el análisis con un indicador que tenga sentido para el problema.

# SOLUCIÓN B — conteo de registros por sector, ANTES de limpieza.
conteo_sector = df.groupBy("sector").count().orderBy("sector")
conteo_sector.show()


In [ ]:
# Comparacion con valores esperados

# Obtener cantidades reales
resultado = {
    r["sector"]: r["count"]
    for r in conteo_sector.collect()
}

# Definir cantidades esperadas
esperado = {
    "Agua": 15,
    "Energía": 7,
    "Residuos": 6
}

# Validar resultados
if resultado == esperado:
    print("VALIDACIÓN CORRECTA: Las cantidades coinciden.")
else:
    print("ADVERTENCIA: Las cantidades no coinciden.")
    print("Esperado:", esperado)
    print("Obtenido:", resultado)

## 7. RETO C: detectar y corregir un error lógico
El siguiente fragmento produce un indicador conceptualmente discutible: **promedia los porcentajes diarios** sin analizar reglas de calidad ni ponderación.

Explica cuándo podría ser engañoso. Propón una alternativa y verifica su resultado.

```python
from pyspark.sql.functions import avg, col
indicador = df.withColumn("cumplimiento", col("valor_observado") / col("meta_diaria"))
indicador.groupBy("sector").agg(avg("cumplimiento")).show()
```

**No ejecutes sin analizar primero** el denominador no válido y los registros duplicados.

In [ ]:
# Probando el codigo propuesto

indicador = df.withColumn("cumplimiento", col("valor_observado") / col("meta_diaria"))
indicador.groupBy("sector").agg(avg("cumplimiento")).show()

In [ ]:
# Mejorando el codigo propuesto
# El cumplimiento se muestra en porcentaje (%)

# Crear indicador de cumplimiento
indicador = df.withColumn(
    "cumplimiento",
    col("valor_observado") / col("meta_diaria")
)

# Calcular cumplimiento promedio por sector
resultado = (
    indicador
    .groupBy("sector")
    .agg(
        round(avg("cumplimiento") * 100,2)
            .alias("cumplimiento_promedio_pct")
    )
    .orderBy("sector")
)

resultado.show(truncate=False)


### Regla de negocio: Depuración y validación de datos para el cálculo del indicador ponderado

Para garantizar la consistencia de los datos utilizados en el cálculo del indicador ponderado, se establece que los registros duplicados deberán identificarse y depurarse, eliminando únicamente las duplicaciones exactas asociadas a un mismo identificador.

Asimismo, se excluirán del cálculo aquellos registros que presenten valores observados nulos o metas diarias menores o iguales a cero, debido a que no permiten calcular válidamente el indicador de cumplimiento.

Los registros cuyo valor observado supere la meta diaria, como el caso **R027**, deberán conservarse, dado que superar una meta no constituye necesariamente un error y puede representar un resultado legítimo de la operación.

**Esta política tiene carácter provisional y fines didácticos.** Antes de aplicarla a datos reales, será necesario validar la definición de la clave de identificación, los criterios de duplicidad y las reglas de negocio que determinan la validez de los registros.

In [ ]:
# RETO C: escribe tu corrección justificada aquí

# Depurando datos
print("Filas originales:",df.count())
base_deduplicada = df.dropDuplicates(["id_registro"])
base_valida = base_deduplicada.filter(
    col("meta_diaria").isNotNull() &
    (col("meta_diaria") > 0) &
    col("valor_observado").isNotNull()
)
print("Filas validas:",base_valida.count())

# Calculando el indicador
resumen = (base_valida.groupBy("sector")
    .agg(spark_sum("valor_observado").alias("total_observado"),
         spark_sum("meta_diaria").alias("total_meta"),
         count("*").alias("registros_validos"))
    .withColumn("cumplimiento_ponderado", col("total_observado") / col("total_meta"))
    .withColumn("cumplimiento_ponderado_pct", round(col("total_observado") / col("total_meta")*100,2))
    .orderBy("sector"))
resumen.show(truncate=False)


## 8. Pruebas de consistencia y reflexión
Antes de concluir, ejecuta la comprobación básica. Esta valida únicamente el archivo de entrenamiento original, **no valida tus decisiones de limpieza**.

In [ ]:
assert df.count() == 28, "El archivo fue alterado o no corresponde al suministrado"
assert len(df.columns) == 7
print("Estructura de archivo de práctica verificada")

**Discusión de cierre (responder con evidencia):**
1. ¿Qué ventaja representa Spark frente a una lectura convencional de CSV en Python? ¿En qué tamaño de datos sería injustificado afirmar que Spark es más rápido?
2. ¿Qué diferencia existe entre una transformación y una acción? Señala una línea de este notebook como evidencia.
3. ¿Por qué un conteo correcto no garantiza calidad de datos?
4. ¿Qué cambió en tu solución después de inspeccionar los registros anómalos?

**Entregables:** notebook ejecutado, cuadro de calidad de datos, criterios de corrección y conclusiones breves.

**Uso de IA:** si se utilizó, declarar herramienta, finalidad, código contrastado y correcciones realizadas; cada integrante debe poder explicar su propuesta.

**Reto opcional:** exporta a CSV los registros observados como anomalías y documenta tu criterio de selección. No sobrescribas el archivo original.